# Worked solutions

This notebook is identical to the student version except that every 🔵 `# TODO` has been filled in, **with commentary on why the answer is what it is** rather than just the code. Read the comments — the reasoning is the point, not the syntax.

Everything else, including the ✏️ YOUR TURN cells, is unchanged: those have no single right answer.


# Module H — Small-molecule chemistry

## Which molecules are worth making in the lab?

### What you will be able to do by the end

1. read a molecule's structure written as text (SMILES) and the numbers computed from it
2. explain what BACE1 is and why it was one of the most-pursued drug targets in Alzheimer's research
3. build models that predict whether a compound inhibits BACE1, from structure alone
4. demonstrate **scaffold leakage** — the chemistry twin of the subject-leakage problem in imaging
5. explain the gap between 'binds the target in a dish' and 'helps a patient'

### The data

**Entirely real.** This is the **MoleculeNet BACE-1 benchmark**: 1513 real compounds with real measured binding affinities (pIC50) against human β-secretase 1, curated from published medicinal chemistry, distributed under the MIT licence. Each compound comes with its structure as a SMILES string and around 590 precomputed molecular descriptors, of which we keep the eleven most interpretable.

*(One thing is computed by us: the `analogue_series` grouping, which stands in for a true chemical scaffold. Section 2 explains why it exists and what it approximates.)*

### How to work through this notebook

Run the cells in order, top to bottom. The notebook is split into four sections:

| | Section | What happens |
|---|---|---|
| 1 | **Understand the data** | Meet every column and every person in the table |
| 2 | **Quality control** | Find the flaws before they fool you |
| 3 | **Build models** | Start from something trivial, then climb |
| 4 | **Read the results** | Turn numbers into a clinical judgement |

Look out for these markers:

- ✏️ **YOUR TURN** — change the value shown, re-run the cell, watch the figure change. Everyone does these.
- 🟢 run and read · 🔵 write a little code · ⚫ take home
- 🧠 a question to think about; the answer is hidden underneath, so try first

**In a hurry?** Run section 1 quickly, then do section 2.3 (the leakage demo) and section 3 properly.

---

*Teaching material. Nothing here is a diagnostic tool, and no result in this notebook is clinical evidence.*


In [ ]:
# Run me first. This finds the project folder, loads the shared helpers,
# and prints exactly where this module's data came from.
from pathlib import Path
import sys
repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').exists())
sys.path.insert(0, str(repo_root / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plots
from data import load_data, load_extra, provenance
from models import split_data, train_model, evaluate, compare_models, sweep_parameter, MODEL_CHOICES

pd.set_option('display.width', 160)
print(provenance('H'))


---
# 1 · Understand the data

### Why BACE1?

The amyloid hypothesis says Alzheimer's begins when amyloid-β accumulates in the brain. Amyloid-β is cut out of a larger protein (APP) by two enzymes in sequence. **β-secretase 1 — BACE1 — makes the first cut.** Block BACE1 and, in principle, you stop amyloid-β being made at all.

This made BACE1 one of the most intensively pursued drug targets in the history of neurology. Billions were spent. Potent, brain-penetrant BACE1 inhibitors were developed, and they worked: they cut amyloid-β production dramatically in humans.

**Then the trials failed.** Verubecestat, lanabecestat, atabecestat — halted, one after another. Several showed patients on the drug declining *faster* than those on placebo. Whether that means the amyloid hypothesis is wrong, or the drugs were given too late, or BACE1 does something else essential we should not have blocked, is still argued.

Hold both halves of that story. The model you build today is a real, useful tool for the step it addresses — and it would have been just as confident about the compounds that failed.


### 1.1 The table

| Column | Meaning |
|---|---|
| `smiles` | The molecule's structure, written as text. `c1ccccc1` is a benzene ring. |
| `pic50` | Measured potency: −log₁₀ of the concentration needed to inhibit half the enzyme. **Higher = more potent.** 9 means nanomolar; 5 means barely active. |
| `active` | 1 if pIC50 ≥ 7. This threshold is a convention, not a law of nature — see 2.1. |
| `analogue_series` | Which family of near-identical molecules this belongs to. Computed by us. |
| `mw` | Molecular weight (daltons). |
| `alogp` | Calculated fat-solubility. Drugs must cross membranes — and, for the brain, the blood–brain barrier. |
| `hbd`, `hba` | Hydrogen-bond donors and acceptors — how it sticks to a protein. |
| `rb` | Rotatable bonds — how floppy it is. |
| `psa` | Polar surface area. Above ~90 Å², getting into the brain becomes hard. |
| `ringcount`, `heavyatomcount`, `chiralcentercount`, `mr`, `polar` | Further shape and size descriptors. |


In [ ]:
df = load_data('H')
print(f'{len(df)} real compounds, {df.active.sum()} of them active (pIC50 >= 7).')
print(f'Grouped into {df.analogue_series.nunique()} analogue series.\n')
display(df.head()[['compound_id', 'pic50', 'active', 'analogue_series', 'mw', 'alogp', 'hbd', 'hba', 'psa']])
print('\nAn example SMILES string:')
print(' ', df.iloc[0]['smiles'])


### 1.2 Potency is continuous; the label is not

**Predict before you run:** where should the active/inactive line go?


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.hist(df['pic50'], bins=45, color='#2c6fbb')
ax.axvline(7.0, color='#c0392b', linewidth=2)
ax.annotate('the "active" cutoff\n(a convention)', (7.0, ax.get_ylim()[1] * 0.75),
            xytext=(10, 0), textcoords='offset points', color='#c0392b', fontsize=9)
ax.set_xlabel('pIC50 (higher = more potent)'); ax.set_ylabel('number of compounds')
ax.set_title('Real measured potencies against BACE1')
plt.tight_layout(); plt.show()

plots.plot_class_balance(df['active'].map({0: 'inactive', 1: 'active'}),
                         title='After applying the cutoff')
plt.show()


### 1.3 ✏️ Your turn — do the simple rules work?

Medicinal chemists have long used quick rules of thumb about which molecules can become drugs — **Lipinski's rule of five** (molecular weight under 500, logP under 5, ≤5 H-bond donors, ≤10 acceptors) being the famous one. Do those numbers separate active BACE1 inhibitors from inactive ones?


In [ ]:
# ==========================================================================
# ✏️  YOUR TURN
#   Change DESCRIPTOR and re-run. Try each of:
#     'mw', 'alogp', 'hbd', 'hba', 'psa', 'rb', 'ringcount'
#   Which single descriptor separates active from inactive best?
#   Is any of them good enough on its own?
# ==========================================================================
DESCRIPTOR = 'mw'

labelled = df.assign(status=df['active'].map({0: 'inactive', 1: 'active'}))
plots.plot_by_group(labelled, DESCRIPTOR, 'status', title=f'{DESCRIPTOR} for active and inactive compounds')
plt.show()

plots.plot_scatter(df[DESCRIPTOR], df['pic50'], colour_by=labelled['status'],
                   xlabel=DESCRIPTOR, ylabel='pIC50 (measured potency)',
                   title=f'Does {DESCRIPTOR} predict potency?', legend_title='class')
plt.show()

correlation = df[DESCRIPTOR].corr(df['pic50'])
print(f'Correlation between {DESCRIPTOR} and potency: {correlation:+.3f}')

lipinski = ((df['mw'] <= 500) & (df['alogp'] <= 5) & (df['hbd'] <= 5) & (df['hba'] <= 10))
print(f"\nCompounds passing Lipinski's rule of five: {lipinski.sum()} of {len(df)}")
print(f'  ... of the ACTIVE compounds:   {lipinski[df.active == 1].mean():.0%} pass')
print(f'  ... of the INACTIVE compounds: {lipinski[df.active == 0].mean():.0%} pass')
print('\nBACE1 inhibitors are famously large and greasy. Rules of thumb are thumbs, not rules.')


🧠 **Think first:** Most known BACE1 inhibitors break Lipinski's rule of five. Should we discard the rule, or discard the compounds?

<details>
<summary>Click for one good answer</summary>

Neither, quite. The rule of five describes what *orally absorbed* drugs have historically looked like — it is a summary of past successes, not a law of chemistry. BACE1's active site is a long groove that needs a long molecule to fill it, so potent inhibitors are large and greasy almost by necessity, and getting them into the brain as well is genuinely hard.

The transferable point: **a rule of thumb learned from one distribution quietly becomes a filter that excludes anything new.** If you had screened with Lipinski as a hard gate, you would have thrown away most of this dataset before modelling it — which is the same failure as a diagnostic model trained on one cohort refusing to work on another.

</details>


### 🎚 Go further — pick whichever suits you

- 🟢 **Everyone:** run 1.3 for every descriptor and find the one with the strongest correlation with potency.
- 🔵 **If you want to write code:** make a scatter plot of `mw` against `alogp` coloured by activity. That plot is called **chemical space**, and it is how chemists picture a compound library.
- ⚫ **Take home:** install RDKit (`pip install rdkit`) and draw a few molecules from their SMILES with `rdkit.Chem.Draw.MolToImage`. Compare the most and least potent compounds by eye.


---
# 2 · Quality control

Chemistry datasets have their own characteristic traps, and one of them is an exact structural twin of the problem in the imaging module.


### 2.1 The cutoff is arbitrary, and it moves your results

Nothing biological happens at pIC50 = 7. A compound at 6.99 and one at 7.01 are indistinguishable in the lab, and yet one is 'active' and the other is not. Accuracy measured near the threshold is measuring the threshold.


In [ ]:
near_line = df[(df['pic50'] > 6.7) & (df['pic50'] < 7.3)]
print(f'{len(near_line)} compounds ({len(near_line) / len(df):.0%}) sit within 0.3 log units of the cutoff.')
print('Assay-to-assay variation is often larger than that.\n')

cutoffs = [6.0, 6.5, 7.0, 7.5, 8.0]
shares = [100 * (df['pic50'] >= cutoff).mean() for cutoff in cutoffs]
plots.plot_score_comparison([f'pIC50 >= {c}' for c in cutoffs], shares,
                            colours=['#2c6fbb'] * len(cutoffs),
                            title='What counts as "active" depends entirely on where you draw the line',
                            ylabel='percent of the library called active')
plt.show()


### 2.2 Analogue series — why these molecules are not independent

Medicinal chemistry does not generate compounds at random. A chemist finds one promising molecule and then makes fifty close relatives, changing one group at a time. Those fifty are **not fifty independent data points** — they are one idea, measured fifty times.

In real practice you group compounds by their **Murcko scaffold**: strip away the side chains and keep the core ring system. That needs RDKit, which we do not require, so `analogue_series` was computed by clustering the full descriptor profile — compounds with near-identical descriptors are near-identical molecules. It is an approximation of the real thing, and the 🔵 extension below computes true scaffolds if you have RDKit installed.


In [ ]:
sizes = df.groupby('analogue_series').size().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.bar(range(len(sizes)), sizes.values, color='#2c6fbb')
ax.set_xlabel('analogue series (sorted by size)'); ax.set_ylabel('compounds in the series')
ax.set_title(f'{len(sizes)} series covering {len(df)} compounds — the biggest has {sizes.max()}')
plt.tight_layout(); plt.show()

activity_by_series = df.groupby('analogue_series')['active'].mean()
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.hist(activity_by_series, bins=20, color='#e08214')
ax.set_xlabel('fraction of the series that is active'); ax.set_ylabel('number of series')
ax.set_title('Series tend to be all-active or all-inactive — that is the leakage risk')
plt.tight_layout(); plt.show()


### 2.3 ✏️ Your turn — scaffold leakage, the chemistry twin of subject leakage

Here is the parallel, made explicit:

| Module A (imaging) | Module H (chemistry) |
|---|---|
| One patient, several scans | One scaffold, several analogues |
| Split by row → the model recognises *the patient* | Split by row → the model recognises *the series* |
| Fix: `groups=subject_id` | Fix: `groups=analogue_series` |
| The lie: it will fail on a new patient | The lie: it will fail on a new chemical series |

**Identical mathematics, two disciplines that rarely talk to each other, and the same fix: one argument.** And in drug discovery the consequence is expensive — a model that only recognises series it has already seen tells you nothing about the genuinely new chemistry you were hoping to find.


In [ ]:
# ==========================================================================
# ✏️  YOUR TURN
#   Flip GROUP_BY_SERIES between True and False and re-run.
#   Then try different models. Which model is flattered most by cheating?
#   (Same question, same answer, as module A section 2.3.)
# ==========================================================================
GROUP_BY_SERIES = True
MODEL = 'random_forest'

descriptors = ['mw', 'alogp', 'hbd', 'hba', 'rb', 'heavyatomcount',
               'chiralcentercount', 'ringcount', 'psa', 'mr', 'polar']
X = df[descriptors]
y = df['active']

both = {}
for label, grouping in [('split by COMPOUND\n(leaky)', None),
                        ('split by SERIES\n(honest)', df['analogue_series'])]:
    X_tr, X_te, y_tr, y_te = split_data(X, y, groups=grouping)
    shared = set(df.loc[X_tr.index, 'analogue_series']) & set(df.loc[X_te.index, 'analogue_series'])
    both[label] = evaluate(train_model(MODEL, X_tr, y_tr), X_te, y_te)['auroc']
    print(f'{label.splitlines()[0]:<22s} {len(shared):>3d} series appear in BOTH halves')

plots.plot_score_comparison(list(both), list(both.values()),
                            colours=['#c0392b', '#2c6fbb'], reference=0.5,
                            title=f'{MODEL}: identical data, identical model, different split',
                            ylabel='AUROC')
plt.show()

grouping = df['analogue_series'] if GROUP_BY_SERIES else None
X_train, X_test, y_train, y_test = split_data(X, y, groups=grouping)
print(f'\nEverything below uses GROUP_BY_SERIES = {GROUP_BY_SERIES}. '
      f'{len(X_train)} train, {len(X_test)} test.')


### 2.4 Activity cliffs

The other thing that makes chemistry hard: two molecules can differ by a single atom and differ by a thousandfold in potency. Chemists call this an **activity cliff**. Any model that assumes "similar molecules have similar activity" — which is essentially all of them — will fall off it.


In [ ]:
spread = df.groupby('analogue_series')['pic50'].agg(['min', 'max', 'count'])
spread['range'] = spread['max'] - spread['min']
cliffs = spread[spread['count'] >= 5].sort_values('range', ascending=False).head(10)

fig, ax = plt.subplots(figsize=(7.5, 4))
positions = np.arange(len(cliffs))
ax.hlines(positions, cliffs['min'], cliffs['max'], color='#2c6fbb', linewidth=3)
ax.scatter(cliffs['min'], positions, color='#c0392b', s=45, zorder=3, label='weakest in series')
ax.scatter(cliffs['max'], positions, color='#5aa469', s=45, zorder=3, label='most potent in series')
ax.set_yticks(positions, [f'series {int(index)} (n={int(row["count"])})' for index, row in cliffs.iterrows()],
              fontsize=8)
ax.set_xlabel('pIC50'); ax.legend(fontsize=9)
ax.set_title('Within one family of near-identical molecules, potency can span 4 log units')
plt.tight_layout(); plt.show()
print('A 4-unit pIC50 range means a 10000-fold difference in potency, within molecules a chemist')
print('would call "the same compound with a tweak". This is why chemistry resists prediction.')


### 2.5 QC verdict

**Usable and genuinely useful, with three rules.**

1. **Always group by `analogue_series`.** Everything below does.
2. The binary label is a convention. Where it matters, model the continuous pIC50 instead (⚫ below).
3. Expect a lower score than the imaging or biomarker modules, and do not read that as failure. Predicting new chemistry is *supposed* to be hard; a model that triages a million compounds down to a thousand worth synthesising has done its job even at AUROC 0.7.

*(**Express path:** you can start from section 3 — run its catch-up cell first and everything below stands alone.)*


---
# 3 · Build models

All models below use the **series-grouped** split, so the held-out compounds come from chemistry the model has never seen. That is the realistic test: can it rank molecules from a *new* series?


### 🚏 Taking the Express path? Run this one cell first

It rebuilds everything sections 3 and 4 need, so you can start here without having run sections 1 and 2 yourself. **If you did run them, run this anyway** — it just redefines the same things and costs a second.


In [ ]:
# Express catch-up: safe to run whether or not you did sections 1 and 2.
df = load_data('H')
descriptors = ['mw', 'alogp', 'hbd', 'hba', 'rb', 'heavyatomcount',
               'chiralcentercount', 'ringcount', 'psa', 'mr', 'polar']
X = df[descriptors]
y = df['active']
print(f'{len(df)} compounds in {df.analogue_series.nunique()} analogue series; '
      f'{int(y.sum())} active. Ready for section 3.')


### 3.1 The ladder


In [ ]:
X_train, X_test, y_train, y_test = split_data(X, y, groups=df['analogue_series'])
print(f'{len(X_train)} training compounds from {df.loc[X_train.index, "analogue_series"].nunique()} series;')
print(f'{len(X_test)} test compounds from {df.loc[X_test.index, "analogue_series"].nunique()} '
      f'completely different series.\n')

ladder = ['baseline', 'logistic', 'knn', 'tree', 'random_forest', 'gradient_boosting', 'svm', 'mlp']
table = compare_models(ladder, X_train, y_train, X_test, y_test)
display(table)
plots.plot_model_comparison(table, metric='auroc',
                            title='Predicting BACE1 activity from 11 descriptors (series-grouped)')
plt.show()


### 3.2 ✏️ Your turn — tune it

Same dial-turning as the other modules, on real chemistry.


In [ ]:
# ==========================================================================
# ✏️  YOUR TURN
#   Pick a model and re-run. Try 'random_forest', 'gradient_boosting',
#   'svm', 'knn', 'tree'.
#   Note how much lower these curves sit than in the biomarker module.
#   That is the difficulty of the problem, not a fault in the code.
# ==========================================================================
MODEL = 'gradient_boosting'

description, parameter, values = MODEL_CHOICES[MODEL]
print(f'{MODEL}: {description}\nSweeping {parameter} over {values}\n')

swept, train_scores, test_scores = sweep_parameter(
    MODEL, parameter, values, X_train, y_train, X_test, y_test)
plots.plot_parameter_sweep(swept, train_scores, test_scores, parameter,
                           title=f'{MODEL} on BACE1 compounds from unseen series')
plt.show()
print(f'Best held-out {parameter}: {values[int(np.argmax(test_scores))]}')


### 3.3 Chemical space

Where do the compounds sit relative to each other, and does the model's opinion follow the chemistry?


In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

space = PCA(n_components=2, random_state=42).fit_transform(
    StandardScaler().fit_transform(df[descriptors].to_numpy()))

plots.plot_scatter(space[:, 0], space[:, 1],
                   colour_by=df['active'].map({0: 'inactive', 1: 'active'}),
                   xlabel='chemical space, axis 1', ylabel='chemical space, axis 2',
                   title='All 1513 compounds, coloured by measured activity', legend_title='class')
plt.show()

final_model = train_model('gradient_boosting', X_train, y_train)
probability = final_model.predict_proba(X_test)[:, 1]

fig, ax = plt.subplots(figsize=(6.6, 4.6))
test_positions = [df.index.get_loc(index) for index in X_test.index]
scatter = ax.scatter(space[test_positions, 0], space[test_positions, 1], c=probability,
                     cmap='coolwarm', s=40, edgecolor='white', linewidth=0.4)
fig.colorbar(scatter, ax=ax, label='predicted probability of being active')
ax.set_xlabel('chemical space, axis 1'); ax.set_ylabel('chemical space, axis 2')
ax.set_title('Held-out compounds, coloured by what the model believes')
plt.tight_layout(); plt.show()


### 3.4 🔵 Your turn to write code — predict potency, not a label

The binary label threw away information (2.1). Real triage ranks compounds by expected potency. Swap the classifier for a **regressor** on `pic50` and see how well the ranking holds up.

> **How practitioners think about this:** in virtual screening you rarely care about accuracy. You care about **enrichment** — if you synthesise the top 100 compounds the model suggests, how many more actives do you get than by picking 100 at random? A model with a mediocre AUROC can still be worth millions if its top slice is good.


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score

potency_train = df.loc[X_train.index, 'pic50']
potency_test = df.loc[X_test.index, 'pic50']

# ✅ Worked solution.
regressor = Pipeline([
    ('scale', StandardScaler()),
    ('model', GradientBoostingRegressor(n_estimators=200, max_depth=3, random_state=42)),
]).fit(X_train, potency_train)
predicted_potency = regressor.predict(X_test)

plots.plot_scatter(potency_test, predicted_potency,
                   xlabel='measured pIC50', ylabel='predicted pIC50',
                   title=f'Predicting potency on unseen series (R2 = {r2_score(potency_test, predicted_potency):.2f})')
plt.plot([4, 10], [4, 10], '--', color='#8a8a8a')
plt.show()

top = np.argsort(predicted_potency)[::-1][:100]
hit_rate = (potency_test.to_numpy()[top] >= 7).mean()
print(f'Top 100 by prediction: {hit_rate:.0%} are truly active.')
print(f'Picking 100 at random: {(potency_test >= 7).mean():.0%} would be.')
print(f'Enrichment factor: {hit_rate / max((potency_test >= 7).mean(), 1e-9):.2f}x')

# How to read the R2, which will be low - possibly near zero or negative.
#
# That is not a broken model, it is the honest difficulty of the task: we are asking it to
# predict potency for chemical series it has never seen, from eleven crude whole-molecule
# descriptors that say nothing about which atoms point where. A real project would use
# fingerprints or a graph network, and would still find cross-series prediction hard.
#
# The enrichment factor is the number that decides whether this is worth doing. Synthesis
# costs thousands per compound, so even a 2x enrichment halves the cost of finding a hit.
# R2 measures whether you can predict the number; enrichment measures whether you can pick
# the winners. In screening, only the second one buys anything.


🧠 **Think first:** The series-grouped AUROC is far lower than the leaky one. Which number should go in the paper — and which describes what the model will do next Monday?

<details>
<summary>Click for one good answer</summary>

The same number answers both: the series-grouped one. The leaky score describes performance on chemistry the model has already seen, and nobody needs a model for that — they can look the answer up.

The reason the leaky number keeps getting published is that it is produced by the *default* behaviour of every machine-learning library. `train_test_split` with no `groups` argument is one keystroke shorter than the correct call, and it silently gives a better answer. Whenever the convenient default flatters you, that is exactly when to check what it assumed — here, that every row is an independent draw, which is false for both molecules and patients.

</details>


### 🎚 Go further — pick whichever suits you

- 🟢 **Everyone:** in 3.2, compare `'gradient_boosting'` and `'knn'`. Why does the nearest-neighbour model do badly on unseen series?
- 🔵 **If you want to write code:** complete 3.4, then compute the enrichment factor for the top 5% instead of the top 100.
- ⚫ **Take home:** install RDKit and replace the eleven descriptors with **Morgan fingerprints** — a 2048-bit vector recording which substructures are present. That is what production models actually use, and it usually adds several AUROC points.


---
# 4 · Read the results

In drug discovery a model's output is not a diagnosis but a **shopping list**: which compounds get made next. So the figures that matter are about the top of the ranking.


### 4.1 The standard views


In [ ]:
predicted = (probability >= 0.5).astype(int)
final_metrics = evaluate(final_model, X_test, y_test)

plots.plot_confusion(y_test, predicted, labels=('inactive', 'active'),
                     title='Held-out compounds from unseen chemical series')
plt.show()
plots.plot_roc_pr(y_test, probability, title='BACE1 activity prediction, series-grouped')
plt.show()
plots.plot_calibration(y_test, probability)
plt.show()
for name, value in final_metrics.items():
    print(f'  {name:<20s} {value:.3f}')


**The errors mean money, not medicine, and they are not symmetric.** A false positive costs a chemist a few weeks and a few thousand euros making something that turns out to be inert — annoying, survivable, and you find out quickly. A **false negative** is a genuinely good compound that never gets made, and nobody ever finds out. That asymmetry is why screening models are usually run at a *low* threshold: cast wide, let the assay do the rejecting.


### 4.2 ✏️ Your turn — how many compounds would you make?

The real decision. You have a synthesis budget. The model ranks the library. How deep do you go?


In [ ]:
# ==========================================================================
# ✏️  YOUR TURN
#   BUDGET is how many compounds you can afford to synthesise.
#   Try 10, 50, 200.
#   Compare your hit rate to the 'pick at random' rate. The ratio
#   between them is the enrichment factor, and it is the only
#   number a project leader actually cares about.
# ==========================================================================
BUDGET = 50

ranking = np.argsort(probability)[::-1]
chosen = ranking[:BUDGET]
truth = y_test.to_numpy()

hit_rate = truth[chosen].mean()
random_rate = truth.mean()

plots.plot_score_comparison(
    [f'your top {BUDGET}', f'{BUDGET} picked at random'],
    [100 * hit_rate, 100 * random_rate], colours=['#2c6fbb', '#8a8a8a'],
    title=f'Enrichment factor: {hit_rate / max(random_rate, 1e-9):.2f}x',
    ylabel='percent of chosen compounds that are truly active')
plt.show()

# The full picture: how the hit rate decays as you go deeper down the ranking.
depths = np.arange(5, len(truth) + 1, 5)
rates = [100 * truth[ranking[:depth]].mean() for depth in depths]
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(depths, rates, color='#2c6fbb', linewidth=2, label='model ranking')
ax.axhline(100 * random_rate, color='#8a8a8a', linestyle='--', label='random picking')
ax.axvline(BUDGET, color='#e08214', linewidth=2, label=f'your budget ({BUDGET})')
ax.set_xlabel('how many compounds you synthesise, in ranked order')
ax.set_ylabel('percent that turn out active')
ax.set_title('The enrichment curve — the actual product of a screening model')
ax.legend(fontsize=9); plt.tight_layout(); plt.show()
print(f'Making the top {BUDGET}: {int(truth[chosen].sum())} real actives found.')
print(f'Making {BUDGET} at random: about {random_rate * BUDGET:.0f} would be expected.')


### 4.3 What is the model looking at?

Feature importance plus exact Shapley values on the six most useful descriptors.


In [ ]:
from interpret import shapley_values, shapley_importance, baseline_prediction

built_in = final_model.named_steps['model'].feature_importances_
names = [name.split('__')[-1] for name in final_model.named_steps['preprocess'].get_feature_names_out()]
plots.plot_importance(names, built_in,
                      title='Gradient boosting: how often each descriptor was used to split',
                      xlabel='built-in importance')
plt.show()

top_six = [names[index] for index in np.argsort(built_in)[::-1][:6]]
print('Explaining the six most-used descriptors with exact Shapley values:', ', '.join(top_six), '\n')
shap_frame = shapley_values(final_model, X_test.head(60), X_train, features=top_six)
importance = shapley_importance(shap_frame)
plots.plot_importance(importance.index, importance.values,
                      title='Average influence on the predicted probability of activity',
                      xlabel='mean |contribution|')
plt.show()

COMPOUND = 0
one = shap_frame.iloc[COMPOUND].sort_values()
plots.plot_importance(one.index, one.values,
                      title=f'{df.loc[X_test.index[COMPOUND], "compound_id"]}: why the model rated it as it did',
                      xlabel='contribution to predicted probability')
plt.show()
print('SMILES:', df.loc[X_test.index[COMPOUND], 'smiles'])
print(f'Measured pIC50: {df.loc[X_test.index[COMPOUND], "pic50"]:.2f} '
      f'({"active" if y_test.iloc[COMPOUND] else "inactive"})')
print(f'Cohort average {baseline_prediction(final_model, X_train):.3f} {one.sum():+.3f} '
      f'= {baseline_prediction(final_model, X_train) + one.sum():.3f}')


### 4.4 The compounds it got wrong


In [ ]:
wrong = np.where(predicted != truth)[0]
confident_and_wrong = wrong[np.argsort(np.abs(probability[wrong] - 0.5))[::-1]][:6]

print('The six compounds the model was most confident about — and wrong about:\n')
for position in confident_and_wrong:
    row = df.loc[X_test.index[position]]
    print(f"  {row['compound_id']:>10s}  model said {probability[position]:.2f}, "
          f"truth: pIC50 {row['pic50']:.2f} ({'active' if row['active'] else 'inactive'})")
    print(f"             {row['smiles'][:90]}")

check = X_test.copy()
check['correct'] = (predicted == truth).astype(int)
check['size_band'] = pd.cut(check['mw'], [0, 400, 500, 600, 2000],
                            labels=['<400', '400-500', '500-600', '600+'])
check['greasiness'] = pd.cut(check['alogp'], [-5, 2, 4, 20], labels=['polar', 'medium', 'greasy'])
for subgroup in ['size_band', 'greasiness']:
    plots.plot_subgroup_errors(check.dropna(subset=[subgroup]), subgroup, 'correct',
                               title=f'Proportion correct by {subgroup}')
    plt.show()


### 4.5 Your headline result


In [ ]:
plots.plot_score_comparison(list(final_metrics), list(final_metrics.values()), reference=0.5,
                            colours=['#2c6fbb'] * 5,
                            title='Module H — gradient boosting, held-out chemical series',
                            ylabel='score')
plt.show()
print(f'{len(df)} real BACE1 compounds; tested on {len(X_test)} from unseen series.')
print(f'Enrichment at the top 50: {truth[ranking[:50]].mean() / max(truth.mean(), 1e-9):.2f}x')
for name, value in final_metrics.items():
    print(f'  {name:<20s} {value:.3f}')


### 4.6 The part that matters most

**What this model does well.** Given a library of a million purchasable compounds, it can rank them in seconds and tell a chemist which thousand to look at. That is a real, deployed, valuable use of machine learning, and it happens every day in every pharmaceutical company.

**What it cannot tell you.**

1. **Binding is not efficacy.** Every compound in this dataset was tested against purified BACE1 in a dish. Nothing here says whether it dissolves, survives the liver, crosses the blood–brain barrier, avoids toxicity, or helps anybody.
2. **The target might be wrong.** This is the BACE1 lesson. The most potent, most selective, most brain-penetrant BACE1 inhibitors ever made — verubecestat, lanabecestat, atabecestat — went into large phase-3 trials in Alzheimer's disease and **failed**. Some arms got worse than placebo. A perfect model of BACE1 inhibition would have ranked those compounds at the very top.
3. **New chemistry is genuinely hard.** Your series-grouped score is the honest one, and it is lower than the leaky one for a reason.

**The transferable point:** *a model can only be as right as the question it was asked.* Every module today optimises something — an AUROC, a gene list, an enrichment factor. None of them optimises "does the patient get better", and no amount of model quality substitutes for choosing the right thing to predict.

---

### 🧠 Final question for the group discussion

Your model would have enthusiastically recommended verubecestat. **Whose job was it to notice that the target was wrong, and what evidence would have changed their mind?** Could any amount of chemistry data have told you?


### 🎚 Go further — pick whichever suits you

- 🟢 **Everyone:** in 4.2, find the budget at which the enrichment curve stops beating random picking.
- 🔵 **If you want to write code:** in 4.3, change `COMPOUND` to one of the confidently-wrong compounds from 4.4 and read its Shapley plot. What misled the model?
- ⚫ **Take home:** read one review of why the BACE1 inhibitor trials failed, and write down which of the three explanations (wrong target / too late / off-target harm) you find most convincing and why.
